# Day 021 — Exercise 4: batch_process_files

**What you'll build:** `batch_process_files(directory, process_fn)` — applies a function to every file's text content and collects `{path, status, result}` dicts. Errors are recorded without stopping the loop.

**Why it matters:** Real file corpora contain malformed, binary, or unexpected files. A resilient batch loop never crashes on a single bad file — it records the error and continues.

In [ ]:
from pathlib import Path

## Your Implementation

In [ ]:
def batch_process_files(directory: str, process_fn) -> list[dict]:
    """
    Apply process_fn to every file's text content in directory.

    Args:
        directory:  Path to the directory to scan.
        process_fn: Callable[[str], Any] — receives file text, returns a result.

    Returns:
        list[dict] — one dict per file:
            On success: {'path': str, 'status': 'ok', 'result': <return value>}
            On error:   {'path': str, 'status': 'error', 'error': str(exception)}
        Subdirectories are skipped. The loop never raises.
    """
    # TODO: results = []
    # TODO: for p in sorted(Path(directory).glob('*')):
    #           if not p.is_file(): continue
    #           try:
    #               content = p.read_text(encoding='utf-8')
    #               result = process_fn(content)
    #               results.append({'path': str(p), 'status': 'ok', 'result': result})
    #           except Exception as e:
    #               results.append({'path': str(p), 'status': 'error', 'error': str(e)})
    # TODO: return results
    pass

## Check Your Work

In [ ]:
import tempfile


def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined
    try:
        assert 'batch_process_files' in globals()
        passed += 1; print('\u2705 Check 1: batch_process_files defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}')
        return

    results = None

    with tempfile.TemporaryDirectory() as tmp:
        td = Path(tmp)
        (td / 'a.txt').write_text('Hello world', encoding='utf-8')
        (td / 'b.txt').write_text('Goodbye world', encoding='utf-8')
        (td / 'binary.dat').write_bytes(b'\x80\x90\xa0')  # invalid utf-8

        # Check 2: returns a list
        try:
            results = batch_process_files(str(td), lambda c: len(c))
            assert isinstance(results, list), \
                f'expected list, got {type(results)}'
            passed += 1; print('\u2705 Check 2: returns a list')
        except Exception as e:
            print(f'\u274c Check 2: {e}')

        # Check 3: each item has path and status keys
        try:
            assert results is not None and len(results) > 0, 'empty results'
            for item in results:
                assert 'path' in item, f"missing 'path': {item}"
                assert 'status' in item, f"missing 'status': {item}"
            passed += 1; print('\u2705 Check 3: each item has path and status keys')
        except Exception as e:
            print(f'\u274c Check 3: {e}')

        # Check 4: successful files have status='ok' and 'result' key
        try:
            assert results is not None
            ok_items = [r for r in results
                        if Path(r['path']).suffix == '.txt']
            assert len(ok_items) == 2, \
                f'expected 2 text files, got {len(ok_items)}'
            for item in ok_items:
                assert item['status'] == 'ok', \
                    f"expected status='ok', got {item['status']!r}"
                assert 'result' in item, \
                    f"missing 'result' in ok item: {item}"
            passed += 1; print("\u2705 Check 4: text files processed with status='ok'")
        except Exception as e:
            print(f'\u274c Check 4: {e}')

        # Check 5: errors recorded without stopping the loop
        try:
            assert results is not None
            statuses = [r['status'] for r in results]
            assert 'ok' in statuses, 'no successful items'
            assert 'error' in statuses, \
                'binary.dat should produce an error entry'
            error_items = [r for r in results if r['status'] == 'error']
            for item in error_items:
                assert 'error' in item, \
                    f"missing 'error' key in error item: {item}"
            passed += 1; print('\u2705 Check 5: errors recorded without stopping loop')
        except Exception as e:
            print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def batch_process_files(directory: str, process_fn) -> list[dict]:
    results = []
    for p in sorted(Path(directory).glob("*")):
        if not p.is_file():
            continue
        try:
            content = p.read_text(encoding="utf-8")
            result = process_fn(content)
            results.append({"path": str(p), "status": "ok", "result": result})
        except Exception as e:
            results.append({"path": str(p), "status": "error", "error": str(e)})
    return results
```

</details>